In [1]:
import ctypes
import os
import io

In [ ]:
class VDIFPacketHeader(ctypes.Structure):
    """Struct representation of a VDIF packet header, dispensing with the bit fields"""

    _fields_ = [
      ("seconds", ctypes.c_uint32),
      ("legacy", ctypes.bool),
      ("invalid", ctypes.bool),
      ("data_frame", ctypes.c_uint32),
      ("ref_epoch", ctypes.c_uint8),
      ("unused", ctypes.c_uint8),
      ("frame_len", ctypes.c_uint32),
      ("log_num_chan", ctypes.c_uint8),
      ("vdif_version", ctypes.c_uint8),
      ("station_id", ctypes.c_uint16),
      ("thread_id", ctypes.c_uint16),
      ("bits_depth", ctypes.c_uint8),
      ("data_type", ctypes.bool),
      ("eud1", ctypes.c_uint32),
      ("edv", ctypes.c_uint8),
      ("eud2", ctypes.c_uint32),
      ("eud3", ctypes.c_uint32),
      ("eud4", ctypes.c_uint32),
    ]

class VDIFPacketRawHeader(ctypes.Structure):
    """Struct representation of a VDIF packet header, just the 32bit ints from file"""

    _fields_ = [ ("word", ctypes_c_uint32 * 8)]

class VDIFPacket(ctypes.Structure):
    """Struct representation of a VDIF packet header"""
    
    @classmethod
    def from_file(cls, filename, max_packets=None):
        """Load a list of VDIFPackets from a kotekan dump file.
        """
        filesize = os.path.getsize(filename)

        buf = bytearray(filesize)

        with io.FileIO(filename, "rb") as fh:
            fh.readinto(buf)

        raw_header = VDIFPacketRawHeader.from_buffer(buf)
        
        header = VDIFPacketHeader()
        header.seconds = (raw_header.word[0] & int('00111111111111111111111111111111', 2)) >> 0
        header.legacy  = (raw_header.word[0] & int('01000000000000000000000000000000', 2)) >> 30
        header.invalid = (raw_header.word[0] & int('10000000000000000000000000000000', 2)) >> 31
        header.data_frame = (raw_header.word[1] & int('00000000111111111111111111111111', 2)) >> 0
        header.ref_epoch  = (raw_header.word[1] & int('00011111000000000000000000000000', 2)) >> 24
        header.unused     = (raw_header.word[1] & int('11100000000000000000000000000000', 2)) >> 30
        header.frame_len    = (raw_header.word[2] & int('00000000111111111111111111111111', 2)) >> 0
        header.log_num_chan = (raw_header.word[2] & int('00011111000000000000000000000000', 2)) >> 24
        header.vdif_version = (raw_header.word[2] & int('11100000000000000000000000000000', 2)) >> 29
        header.station_id =   (raw_header.word[3] & int('00000000000000001111111111111111', 2)) >> 0
        header.thread_id  =   (raw_header.word[3] & int('00000011111111110000000000000000', 2)) >> 16
        header.bits_depth =   (raw_header.word[3] & int('01111100000000000000000000000000', 2)) >> 26
        header.data_type  =   (raw_header.word[3] & int('10000000000000000000000000000000', 2)) >> 31
        header.eud1 = (raw_header.word[4] & int('00000000111111111111111111111111', 2)) >> 0
        header.edv  = (raw_header.word[4] & int('11111111000000000000000000000000', 2)) >> 24
        header.eud2 = raw_header.word[5]
        header.eud3 = raw_header.word[6]
        header.eud4 = raw_header.word[7]
        
        struct_name = "VDIFPacket_" + filename
        struct = type(struct_name, (VDIFPacket,), {})
        struct._fields_ = [
            ("header", VDIFPacketHeader),
            # 32 is size of VDIF header (8 words of 32bits each)
            ("data", ctypes.c_uint8 * (header.frame_len * 8 - 32)),
        ]

        npkts = (len(buf) - 4) // ctypes.sizeof(struct)
        if max_packets:
            npkts = min(npkts, max_packets)
        return (struct * npkts).from_buffer(buf[4:])
